# 07b — Large-pool policy comparison across ten seeds

Compare random queries, confidence triage and two channels with the same
1,000-query budget and the same seed-specific evaluation splits. Reuse cached
trajectories when present and `RECOMPUTE` is false. The cold-start model and
descriptors are needed only when generating new trajectories.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, time, warnings
warnings.filterwarnings('ignore', message='.*sklearn\\.utils\\.parallel\\.delayed.*', category=UserWarning)
from pathlib import Path
raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
assert (raiz / 'config.py').exists(), 'Open this notebook inside the project directory'
os.chdir(raiz); sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import config
from src.models import classification
from src.models import anomalies as an
from src.models import active_learning as al
from src.models import evaluation
from src import pipeline
RS = config.RANDOM_STATE

BATCH_COMB = 99
FRACS    = (0.0, 0.3, 0.6, 1.0)
SEEDS = tuple(range(10))          
print('config · batch:', BATCH_COMB, '· fracs:', FRACS, '· seeds:', SEEDS)


from matplotlib import rcParams
rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 12,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 12,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.frameon': False,
    'legend.fontsize': 11, 'axes.axisbelow': True,
})
C = {'dos': '#2166ac', 'triage': '#d6604d', 'azar': '#9aa0a6'}
FRAC_REP = 0.6


## 1. Load the combined pool or cached comparison


In [ ]:
RECOMPUTE = False  
VERSION_RESULTADOS_ESCALA = 'e2gp-2.0.0'
CSV10 = config.DIR_PROCESSED / 'dos_canales_batch99_10seeds.csv'
CSV_TRI10 = config.DIR_PROCESSED / 'triaje_batch99_10seeds.csv'

if not RECOMPUTE and not (CSV10.exists() and CSV_TRI10.exists()):
    raise FileNotFoundError('Cached trajectories are missing. Import saved results or explicitly set RECOMPUTE=True to repeat AL.')

if CSV10.exists() and CSV_TRI10.exists() and not RECOMPUTE:
    print('Saved per-seed results found: reusing trajectories without repeating Active Learning.')
else:
    tr = classification.features_e2gp('train')
    X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
    batch_paths = pipeline.select_batch(BATCH_COMB)
    assert batch_paths.features_gp2d_test_csv.exists(), 'Combined-pool features are missing; run notebook 07 first'
    te = classification.features_e2gp('test', batch=BATCH_COMB)
    metadata_table = pd.read_csv(batch_paths.test_metadata_clean_csv)
    te = te.merge(metadata_table[['object_id', 'true_target']], on='object_id')
    tt = te['true_target'].to_numpy()
    X_te, _ = classification.e2gp_matrix(te, COLS)
    clf0 = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
    gp_te = te[['object_id']].merge(pd.read_csv(batch_paths.features_gp_test_csv), on='object_id', how='left')
    st_te = te[['object_id']].merge(pd.read_csv(batch_paths.features_test_csv), on='object_id', how='left')
    S = np.column_stack([an.confidence_novelty_score(clf0.predict_proba(X_te)),
                         an.score_isolation_forest(X_te),
                         an.gp_residual_score(gp_te),
                         an.amplitude_score(st_te)])
    desc_te = S[:, 1:]
    con = tt < 990
    v, c = np.unique(tt[con], return_counts=True); singles = set(v[c < 2].tolist())
    keep = ~(con & np.isin(tt, list(singles)))
    X_te, tt, desc_te = X_te[keep], tt[keep], desc_te[keep]
    def make_model(y_lab):
        return classification.e2gp_al(y=y_lab, random_state=RS)
    print(f'pool preparado: {len(tt):,} objects · {int((tt>=990).sum())} anomalies 99x')


## 2. Two-channel trajectories


In [ ]:
if CSV10.exists() and not RECOMPUTE:
    hist10 = pd.read_csv(CSV10)
    print(f'two channels: cached trajectories ({len(hist10):,} rows)')
else:
    t = time.time()
    hist10 = al.two_channel_multiseed(
        X_tr, y_tr, X_te, tt, desc_te, make_model,
        fracs=FRACS, random_seeds=SEEDS, conf_dudoso=0.60, reg_C=0.1,
        budget=1000, b_iter=100, n_iter=30, output_path=str(CSV10), verbose=True)
    print(f'\ndos canales (10 seeds) hecho in {(time.time()-t)/60:.1f} min')

fin10 = hist10.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
resumen10 = (fin10.groupby('frac_desc')
             .agg(MF1_mean=('MF1', 'mean'), MF1_std=('MF1', 'std'),
                  descub_mean=('n_descubiertas', 'mean'), descub_std=('n_descubiertas', 'std'),
                  descub_desc=('descub_desc', 'mean'), descub_clas=('descub_clas', 'mean')).round(3))
display(resumen10)


## 3. Triage and random trajectories


In [ ]:
if CSV_TRI10.exists() and not RECOMPUTE:
    hist_tri10 = pd.read_csv(CSV_TRI10)
    print(f'triage and random: cached trajectories ({len(hist_tri10):,} rows)')
else:
    t = time.time()
    hist_tri10 = al.confidence_triage_multiseed(
        X_tr, y_tr, X_te, tt, make_model,
        modes_list=('triage', 'random'), random_seeds=SEEDS,
        conf_anom=0.30, conf_dudoso=0.60, orden_dudoso='entropy',
        budget=1000, b_iter=100, n_iter=30, output_path=str(CSV_TRI10), verbose=True)
    print(f'\ntriaje+random (10 seeds) hecho in {(time.time()-t)/60:.1f} min')

fin_tri10 = hist_tri10.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
resumen_tri10 = (fin_tri10.groupby('modo')
                 .agg(MF1_mean=('MF1', 'mean'), MF1_std=('MF1', 'std'),
                      descub_mean=('n_descubiertas', 'mean'), descub_std=('n_descubiertas', 'std'),
                      q_anom=('q_anom', 'mean'), n_auto=('n_auto', 'mean')).round(3))
print('Triage by confidence and random (baseline) · 10 seeds:')
display(resumen_tri10)


## 4. Compare the three policies


In [ ]:


dc = pd.read_csv(config.DIR_PROCESSED / 'dos_canales_batch99_10seeds.csv')
tr = pd.read_csv(config.DIR_PROCESSED / 'triaje_batch99_10seeds.csv')
fuentes = [('Two channels', dc[dc.frac_desc == FRAC_REP], C['dos'], 'o'),
           ('Triage by confidence', tr[tr.modo == 'triage'], C['triage'], 's'),
           ('Random sampling', tr[tr.modo == 'random'], C['azar'], '^')]

def agg(df, col):
    g = df.groupby('n_consultas')[col].agg(['mean', 'std', 'size'])
    return g.index.values, g['mean'].values, (g['std'] / np.sqrt(g['size'])).values

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, col, ylab, ttl in [(axes[0], 'recall_anom', 'Recall of anomalies (class 99x)', 'Anomaly discovery'),
                           (axes[1], 'MF1', 'Macro-F1 (known classes)', 'Classification')]:
    for item_name, df, color, mk in fuentes:
        x, m, e = agg(df, col)
        ax.plot(x, m, marker=mk, ms=5, color=color, lw=2.2, label=item_name,
                markeredgecolor='white', markeredgewidth=0.8, zorder=3)
        ax.fill_between(x, m - e, m + e, color=color, alpha=0.18, zorder=1)
    ax.set(xlabel='Cumulative expert queries', ylabel=ylab, title=ttl)
    ax.legend(loc='upper left')
fig.suptitle('Label efficiency in the combined pool (~350k) · 10 seeds', y=1.02)
fig.tight_layout()
ruta_png = config.FIGURE_DIR / 'curvas_aprendizaje_batch99.png'
fig.savefig(ruta_png, dpi=150, bbox_inches='tight')
with Image.open(ruta_png) as im:
    im.convert('RGB').save(config.FIGURE_DIR / 'curvas_aprendizaje_batch99.pdf', resolution=300.0)
plt.show()


In [ ]:
records_list = []
for m in ['triage', 'random']:
    if m in resumen_tri10.index:
        etq = 'Triage by confidence' if m == 'triage' else 'random (baseline)'
        records_list.append({'sistema': etq, 'MF1': resumen_tri10.loc[m, 'MF1_mean'],
                      'MF1_std': resumen_tri10.loc[m, 'MF1_std'],
                      'descub': resumen_tri10.loc[m, 'descub_mean'], 'descub_std': resumen_tri10.loc[m, 'descub_std']})
for fr in resumen10.index:
    records_list.append({'sistema': f'Two channels (frac_desc={fr})', 'MF1': resumen10.loc[fr, 'MF1_mean'],
                  'MF1_std': resumen10.loc[fr, 'MF1_std'],
                  'descub': resumen10.loc[fr, 'descub_mean'], 'descub_std': resumen10.loc[fr, 'descub_std']})
comp10 = pd.DataFrame(records_list)
comp10.to_csv(config.DIR_PROCESSED / 'comparacion_batch99_10seeds.csv', index=False)
print('Combined pool (~350k) · ten seeds · mean ± standard deviation:')
display(comp10)


### 4.1. Paired comparison at discovery allocation 0.6

Use matched seed differences, bootstrap intervals, win counts, rank-biserial
effect sizes and two-sided Wilcoxon tests with Holm adjustment across both
objectives. These intervals describe variability across seeds on the fixed pool.


In [ ]:
n = len(SEEDS)
comp10['MF1_ee'] = (comp10['MF1_std'] / np.sqrt(n)).round(4)
comp10['descub_ee'] = (comp10['descub_std'] / np.sqrt(n)).round(2)
print('Table with standard errors (std/√n):')
display(comp10[['sistema', 'MF1', 'MF1_ee', 'descub', 'descub_ee']])

tri = fin_tri10[fin_tri10['modo'] == 'triage']
dos = fin10[fin10['frac_desc'] == 0.6]
estadistica = evaluation.paired_comparison_summary(
    dos, tri, ['MF1', 'recall_anom'], n_boot=20_000, seed_value=RS)
print('Two channels (frac_desc=0.6) − triage · paired seed comparison:')
display(estadistica.style.format({
    'media_propuesto': '{:.4f}', 'media_referencia': '{:.4f}',
    'diferencia_media': '{:+.4f}', 'ic95_inf': '{:+.4f}', 'ic95_sup': '{:+.4f}',
    'diferencia_mediana': '{:+.4f}', 'rango_biserial': '{:.3f}',
    'p_wilcoxon': '{:.4f}', 'p_holm': '{:.4f}'}))


### Statistical interpretation

Discovery improves in all ten seeds and remains significant after Holm
adjustment (approximately 0.004). Classification improves on average and in
nine seeds, but its adjusted p-value is approximately 0.064. Do not present
that classification difference as statistically conclusive.


## 5. Compare individual-batch and combined-pool regimes


### 5.1. Load and aggregate the saved trajectories


In [ ]:
def agregar(dc_csv, tri_csv, regimen):
    dc = pd.read_csv(config.DIR_PROCESSED / dc_csv)
    tr = pd.read_csv(config.DIR_PROCESSED / tri_csv)
    fdc = dc.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
    ftr = tr.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
    records_list = []
    def add(item_name, s):
        nn = s['semilla'].nunique()
        records_list.append({'sistema': item_name, 'regimen': regimen,
                      'MF1': s['MF1'].mean(), 'MF1_ee': s['MF1'].std()/np.sqrt(nn),
                      'recall': s['recall_anom'].mean(), 'recall_ee': s['recall_anom'].std()/np.sqrt(nn)})
    add('Triage by confidence', ftr[ftr['modo'] == 'triage'])
    add('Random sampling', ftr[ftr['modo'] == 'random'])
    add(f'Two channels (frac_desc={FRAC_REP})', fdc[fdc['frac_desc'] == FRAC_REP])
    return pd.DataFrame(records_list)

b4  = agregar('dos_canales_batch4_10seeds.csv',  'triaje_batch4_10seeds.csv',  'Batch (~33k)')
esc = agregar('dos_canales_batch99_10seeds.csv', 'triaje_batch99_10seeds.csv', 'Combined pool (~350k)')
all_records = pd.concat([b4, esc], ignore_index=True)
print('loaded')


### 5.2. Plot the transition between regimes


In [ ]:
from matplotlib import ticker
sistemas = [(f'Two channels (frac_desc={FRAC_REP})', 'Two channels', C['dos'], '-', 2.6, 'o'),
            ('Triage by confidence',                 'Triage by confidence', C['triage'], '-', 2.6, 's'),
            ('Random sampling',                   'Random sampling (baseline)', C['azar'], '--', 1.6, '^')]
etiquetar = {'Two channels', 'Triage by confidence'}  
regs = ['Batch (~33k)', 'Combined pool (~350k)']
xr = [0, 1]
rec_dos_escala = all_records[(all_records.sistema == f'Two channels (frac_desc={FRAC_REP})') & (all_records.regimen == 'Combined pool (~350k)')]['recall'].iloc[0]
rec_azar_escala = all_records[(all_records.sistema == 'Random sampling') & (all_records.regimen == 'Combined pool (~350k)')]['recall'].iloc[0]
factor_azar_escala = rec_dos_escala / rec_azar_escala

def serie(sis, col):
    return [(all_records[(all_records.sistema==sis)&(all_records.regimen==r)][col].iloc[0],
             all_records[(all_records.sistema==sis)&(all_records.regimen==r)][col+'_ee'].iloc[0]) for r in regs]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
for ax, col, ylab, ttl, logy in [
        (axes[0], 'MF1', 'Macro-F1 (known classes)', 'Classification', False),
        (axes[1], 'recall', 'Anomaly recall (class 99x, log scale)', 'Anomaly discovery', True)]:
    for sis, et, c, ls, lw, mk in sistemas:
        s = serie(sis, col); y = [v[0] for v in s]; e = [v[1] for v in s]
        ax.plot(xr, y, ls, color=c, lw=lw, marker=mk, ms=8, label=et, zorder=3,
                markeredgecolor='white', markeredgewidth=1.1)
        ax.errorbar(xr, y, yerr=e, fmt='none', ecolor=c, elinewidth=1.2, capsize=4, alpha=.8, zorder=2)
        if et in etiquetar:
            for xi, yi in zip(xr, y):
                dy = 10 if col == 'recall' and xi == 0 and et == 'Triage by confidence' else (-12 if col == 'recall' and xi == 0 and et == 'Two channels' else 3)
                ax.annotate(f'{yi:.3f}', (xi, yi), textcoords='offset points',
                            xytext=(-14 if xi == 0 else 14, dy), ha='right' if xi == 0 else 'left',
                            fontsize=10, color=c, fontweight='bold')
    if logy:                                       
        ax.set_yscale('log'); ax.set_ylim(0.003, 0.13)
        ax.set_yticks([0.005, 0.01, 0.02, 0.05, 0.1])
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_xlim(-0.55, 1.55); ax.set_xticks(xr); ax.set_xticklabels(regs)
    ax.set_ylabel(ylab); ax.set_title(ttl); ax.grid(axis='x', alpha=0); ax.margins(y=0.12)

axes[1].annotate(f'×{factor_azar_escala:.1f} over random sampling', xy=(1, rec_dos_escala), xytext=(0.63, 0.068), fontsize=9.5, color=C['dos'],
                 ha='center', arrowprops=dict(arrowstyle='->', color=C['dos'], lw=1.2))
axes[0].legend(loc='lower left')
fig.suptitle('Performance by evaluation regime · one batch vs eleven combined batches',
             fontsize=13.5, fontweight='bold', y=1.02)
fig.text(0.5, -0.02, 'Mean ± standard error (10 seeds). Log-scale recall highlights the improvement in the '
         f'combined pool. Triage approaches random sampling; two channels retain {factor_azar_escala:.1f}× the recall of random sampling.',
         ha='center', fontsize=9.5, style='italic', color='#444')
fig.tight_layout()
ruta_png = config.FIGURE_DIR / 'fig_batch_vs_escala.png'
fig.savefig(ruta_png, bbox_inches='tight')
with Image.open(ruta_png) as im:
    im.convert('RGB').save(config.FIGURE_DIR / 'fig_batch_vs_escala.pdf', resolution=300.0)
plt.show()
print('saved: fig_batch_vs_escala.png / .pdf')


### 5.3. Report means and standard errors


In [ ]:
def celda(m, e): return f'{m:.3f} $\\pm$ {e:.3f}'
sort_order = [f'Two channels (frac_desc={FRAC_REP})', 'Triage by confidence', 'Random sampling']


records_list = []
for sis in sort_order:
    r4 = all_records[(all_records.sistema==sis)&(all_records.regimen=='Batch (~33k)')].iloc[0]
    re = all_records[(all_records.sistema==sis)&(all_records.regimen=='Combined pool (~350k)')].iloc[0]
    records_list.append({'Sistema': sis, 'MF1 (batch)': celda(r4.MF1, r4.MF1_ee), 'MF1 (escala)': celda(re.MF1, re.MF1_ee),
                  'Recall (batch)': celda(r4.recall, r4.recall_ee), 'Recall (escala)': celda(re.recall, re.recall_ee)})
result_table = pd.DataFrame(records_list).set_index('Sistema')
display(result_table)


### Interpretation

At scale, the mean discovery totals are 38.0 for two channels, 12.5 for triage
and 7.0 for random queries. Triage remains above random discovery; the evidence
does not establish mathematical convergence to random sampling. The comparison
changes both pool volume and accumulated heterogeneity under a fixed budget.


In [ ]:
# The fine sweep is a separate experiment, not part of this cached review.
fine_path = config.DIR_PROCESSED / 'dos_canales_batch99_fino_10s.csv'
if not fine_path.exists():
    raise FileNotFoundError('Run scripts/regenerate_fine_sweep_10seeds.py first')
display(pd.read_csv(fine_path).groupby('frac_desc')['semilla'].nunique().rename('seeds'))
